# Data Augmentation

In [ ]:
import os
import glob
import librosa
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Library Augmentasi siap!')

In [ ]:
def add_noise(data, noise_factor=0.005):
    noise = np.random.randn(len(data))
    return data + noise_factor * noise

def pitch_shift(data, sampling_rate, pitch_factor):
    return librosa.effects.pitch_shift(y=data, sr=sampling_rate, n_steps=pitch_factor)

def time_stretch(data, rate):
    return librosa.effects.time_stretch(y=data, rate=rate)

In [ ]:
prep_dir = 'dataset/preprocessed'
aug_dir = 'dataset/augmented'
classes = ['horas', 'sampurasun', 'adilkatalino', 'wawawa', 'kulanuwun', 'tabea', 'apahabarpian', 'noise']
file_counts = {c: 0 for c in classes}
for cls in classes:
    os.makedirs(os.path.join(aug_dir, cls), exist_ok=True)
    files = glob.glob(os.path.join(prep_dir, cls, '*.wav'))
    for file_path in files:
        y, sr = librosa.load(file_path, sr=16000)
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        sf.write(os.path.join(aug_dir, cls, f"{base_name}_ori.wav"), y, sr)
        sf.write(os.path.join(aug_dir, cls, f"{base_name}_noise.wav"), add_noise(y), sr)
        sf.write(os.path.join(aug_dir, cls, f"{base_name}_pitchup.wav"), pitch_shift(y, sr, 2), sr)
        sf.write(os.path.join(aug_dir, cls, f"{base_name}_pitchdown.wav"), pitch_shift(y, sr, -2), sr)
        sf.write(os.path.join(aug_dir, cls, f"{base_name}_fast.wav"), time_stretch(y, 1.2), sr)
        file_counts[cls] += 5
print('PROSES AUGMENTASI SELESAI!')

### Tinjauan Hasil: Efek Augmentasi

In [ ]:
sample_files = glob.glob('dataset/augmented/*/*_ori.wav')
if sample_files:
    sample_ori = sample_files[0]
    base_name = os.path.basename(sample_ori).replace('_ori.wav', '')
    cls_name = os.path.basename(os.path.dirname(sample_ori))
    dir_path = os.path.dirname(sample_ori)
    
    y_ori, sr = librosa.load(sample_ori, sr=16000)
    y_noise, _ = librosa.load(os.path.join(dir_path, f"{base_name}_noise.wav"), sr=sr)
    y_up, _ = librosa.load(os.path.join(dir_path, f"{base_name}_pitchup.wav"), sr=sr)
    y_down, _ = librosa.load(os.path.join(dir_path, f"{base_name}_pitchdown.wav"), sr=sr)
    y_fast, _ = librosa.load(os.path.join(dir_path, f"{base_name}_fast.wav"), sr=sr)
    
    plt.figure(figsize=(15, 10))
    plt.subplot(3, 2, 1)
    librosa.display.waveshow(y_ori, sr=sr, alpha=0.7)
    plt.title('1. Original (Asli)')
    plt.subplot(3, 2, 2)
    librosa.display.waveshow(y_noise, sr=sr, alpha=0.7, color='orange')
    plt.title('2. + Noise')
    plt.subplot(3, 2, 3)
    librosa.display.waveshow(y_up, sr=sr, alpha=0.7, color='green')
    plt.title('3. Pitch Naik')
    plt.subplot(3, 2, 4)
    librosa.display.waveshow(y_down, sr=sr, alpha=0.7, color='purple')
    plt.title('4. Pitch Turun')
    plt.subplot(3, 2, 5)
    librosa.display.waveshow(y_fast, sr=sr, alpha=0.7, color='red')
    plt.title('5. Dipercepat (Time Stretch)')
    plt.tight_layout()
    plt.show()
else:
    print('Belum ada file di dataset/augmented.')

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=list(file_counts.keys()), y=list(file_counts.values()), palette='viridis')
plt.title('Distribusi Total Dataset Setelah Data Augmentation')
plt.xlabel('Kelas (Bahasa Daerah)')
plt.ylabel('Jumlah Rekaman Audio')
plt.xticks(rotation=45)
plt.show()